Compute running balance,reset when account closes.

In [0]:
CREATE OR REPLACE TABLE transactions (
  account_id STRING,
  txn_date DATE,
  amount DECIMAL(10,2),
  status STRING
);

INSERT INTO transactions VALUES
  ('ACC1', '2024-01-01', 100.00, 'open'),
  ('ACC1', '2024-01-02',  50.00, 'open'),
  ('ACC1', '2024-01-03',   0.00, 'closed'),
  ('ACC1', '2024-01-04',  75.00, 'open'),
  ('ACC2', '2024-01-01', 200.00, 'open'),
  ('ACC2', '2024-01-05', -30.00, 'open'),
  ('ACC2', '2024-01-06',   0.00, 'closed'),
  ('ACC2', '2024-01-07', 120.00, 'open');

In [0]:
WITH base AS (
  SELECT
    account_id,
    txn_date,
    amount,
    status,
    SUM(CASE WHEN status = 'closed' THEN 1 ELSE 0 END) OVER (
      PARTITION BY account_id
      ORDER BY txn_date
    ) AS reset_grp
  FROM transactions
)
SELECT
  account_id,
  txn_date,
  amount,
  SUM(amount) OVER (
    PARTITION BY account_id, reset_grp
    ORDER BY txn_date
  ) AS running_balance
FROM base;
